# BookXcess Scraper — Full Inventory Run

Pulls every product from every sub-sitemap on `bookxcess.com`, no limit.

**13 raw columns :** Product ID, Title, Product Type, Vendor, SKU, Current Price, Compare at Price, Currency, Grams, Tags, Body HTML, Image URL, Created At.

In [11]:
!pip install curl_cffi beautifulsoup4 pandas lxml

In [1]:
from curl_cffi import requests as cf_requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import unquote_plus
import time, random, re, pandas as pd, os, threading

SITEMAP_INDEX_URL = "https://www.bookxcess.com/sitemap.xml"
OUTPUT_FILE       = "bookxcess_full_inventory.csv"
PARTIAL_FILE      = "bookxcess_full_inventory_partial.csv"
FAILED_FILE       = "bookxcess_failed_urls.csv"
IMPERSONATE       = "chrome124"
MAX_WORKERS       = 40            # concurrent product fetches
REQUEST_JITTER    = (0.05, 0.2)   # per-thread sleep before each request
MAX_RETRIES       = 3             # retries on 503 / 429 / connection error
RETRY_BACKOFF     = (1.5, 3.5)    # base seconds for exponential backoff jitter
FLUSH_EVERY       = 5_000         # write partial CSV every N successful products

print("Imports OK. Workers:", MAX_WORKERS, "| Retries:", MAX_RETRIES, "| Flush every:", FLUSH_EVERY)

C:\Users\Owner\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Owner\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Imports OK. Workers: 40 | Retries: 3 | Flush every: 5000


## Connection Test

In [2]:
with cf_requests.Session(impersonate=IMPERSONATE) as _s:
    _r = _s.get(SITEMAP_INDEX_URL, timeout=30)

print(f"Status : {_r.status_code}")
print(f"Bytes  : {len(_r.content):,}")
print(f"Snippet: {_r.text[:200]}")
assert _r.status_code == 200, f"Connection failed: HTTP {_r.status_code}"

Status : 200
Bytes  : 8,805
Snippet: <?xml version="1.0" encoding="UTF-8"?>
<sitemapindex xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
  <!-- This is the parent sitemap linking to additional sitemaps for products, collections and


## Step 1 — Collect Every Product URL

In [3]:
def extract_product_locs(xml_text: str) -> list:
    """Return only the page-level <loc> URLs that point to /products/ pages."""
    soup = BeautifulSoup(xml_text, "xml")
    urls = []
    for url_tag in soup.find_all("url"):
        loc = url_tag.find("loc", recursive=False)
        if loc and loc.text and "/products/" in loc.text:
            urls.append(loc.text.strip())
    return urls


def get_all_product_urls(session: cf_requests.Session) -> list:
    resp = session.get(SITEMAP_INDEX_URL, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "xml")
    sub_sitemaps = [
        tag.text.strip()
        for tag in soup.find_all("loc")
        if "products" in tag.text
    ]
    print(f"Found {len(sub_sitemaps)} product sub-sitemap(s).")

    product_urls = []
    for i, sitemap_url in enumerate(sub_sitemaps, 1):
        try:
            r = session.get(sitemap_url, timeout=30)
            r.raise_for_status()
            urls = extract_product_locs(r.text)
            product_urls.extend(urls)
            if i % 10 == 0 or i == len(sub_sitemaps):
                print(f"  [{i}/{len(sub_sitemaps)}] total URLs: {len(product_urls):,}")
        except Exception as exc:
            print(f"  Warning [{i}]: {sitemap_url.split('?')[0]} — {exc}")

        time.sleep(random.uniform(1.0, 1.8))

    # De-duplicate while preserving order
    seen, deduped = set(), []
    for u in product_urls:
        if u not in seen:
            seen.add(u)
            deduped.append(u)
    print(f"\nUnique product URLs: {len(deduped):,} (dropped {len(product_urls) - len(deduped):,} duplicates)")
    return deduped

In [6]:
discovery_session = cf_requests.Session(impersonate=IMPERSONATE)
product_urls = get_all_product_urls(discovery_session)
discovery_session.close()

Found 62 product sub-sitemap(s).
  [10/62] total URLs: 25,000
  [20/62] total URLs: 50,000
  [30/62] total URLs: 75,000
  [40/62] total URLs: 100,000
  [50/62] total URLs: 125,000
  [60/62] total URLs: 150,000
  [62/62] total URLs: 154,714

Unique product URLs: 154,714 (dropped 0 duplicates)


## Step 2 — Concurrent Product Scrape

Each worker thread keeps its own `curl_cffi` session. Throttling errors (503/429) are retried with exponential backoff.

In [4]:
_thread_local = threading.local()

def get_session() -> cf_requests.Session:
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = cf_requests.Session(impersonate=IMPERSONATE)
        _thread_local.session = s
    return s


TRANSIENT_STATUSES = (429, 500, 502, 503, 504)

def fetch_product_data(product_url: str) -> dict:
    """Fetch one product, retrying transient errors. Returns 13 raw fields."""
    json_url = product_url.rstrip("/") + ".json"
    last_exc = None

    for attempt in range(MAX_RETRIES):
        try:
            resp = get_session().get(json_url, timeout=30)
            if resp.status_code in TRANSIENT_STATUSES:
                raise RuntimeError(f"HTTP {resp.status_code}")
            resp.raise_for_status()

            product = resp.json()["product"]
            variant = product["variants"][0] if product.get("variants") else {}
            image   = product.get("image") or {}

            return {
                "Product ID"       : product.get("id"),
                "Title"            : product.get("title"),
                "Product Type"     : product.get("product_type"),
                "Vendor"           : product.get("vendor"),
                "SKU"              : variant.get("sku"),
                "Current Price"    : variant.get("price"),
                "Compare at Price" : variant.get("compare_at_price"),
                "Currency"         : variant.get("price_currency"),
                "Grams"            : variant.get("grams"),
                "Tags"             : product.get("tags"),
                "Body HTML"        : product.get("body_html"),
                "Image URL"        : image.get("src"),
                "Created At"       : product.get("created_at"),
            }
        except Exception as exc:
            last_exc = exc
            if attempt < MAX_RETRIES - 1:
                backoff = random.uniform(*RETRY_BACKOFF) * (2 ** attempt)
                time.sleep(backoff)

    raise last_exc


def fetch_with_jitter(product_url: str) -> dict:
    time.sleep(random.uniform(*REQUEST_JITTER))
    return fetch_product_data(product_url)

In [9]:
results       = []
failed_urls   = []
total         = len(product_urls)
start         = time.time()
flush_lock    = threading.Lock()

def flush_partial():
    with flush_lock:
        pd.DataFrame(results).to_csv(PARTIAL_FILE, index=False, encoding="utf-8-sig")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(fetch_with_jitter, u): u for u in product_urls}

    for i, fut in enumerate(as_completed(futures), 1):
        url = futures[fut]
        try:
            results.append(fut.result())
        except Exception as exc:
            failed_urls.append((url, str(exc)))

        if i % 500 == 0:
            elapsed = time.time() - start
            rate    = i / elapsed
            eta_min = (total - i) / rate / 60 if rate > 0 else 0
            print(f"Scraped {i:,} / {total:,}  |  fails: {len(failed_urls)}  |  {rate:.1f} req/s  |  ETA: {eta_min:.1f} min")

        if i % FLUSH_EVERY == 0:
            flush_partial()
            print(f"  ... flushed partial CSV ({len(results):,} rows)")

elapsed = time.time() - start
print(f"\nFinished in {elapsed/60:.1f} min. Success: {len(results):,}  Failures: {len(failed_urls):,}")

Scraped 500 / 154,714  |  fails: 0  |  107.9 req/s  |  ETA: 23.8 min
Scraped 1,000 / 154,714  |  fails: 0  |  133.3 req/s  |  ETA: 19.2 min
Scraped 1,500 / 154,714  |  fails: 0  |  144.8 req/s  |  ETA: 17.6 min
Scraped 2,000 / 154,714  |  fails: 0  |  152.4 req/s  |  ETA: 16.7 min
Scraped 2,500 / 154,714  |  fails: 0  |  156.5 req/s  |  ETA: 16.2 min
Scraped 3,000 / 154,714  |  fails: 0  |  160.0 req/s  |  ETA: 15.8 min
Scraped 3,500 / 154,714  |  fails: 0  |  162.4 req/s  |  ETA: 15.5 min
Scraped 4,000 / 154,714  |  fails: 0  |  163.8 req/s  |  ETA: 15.3 min
Scraped 4,500 / 154,714  |  fails: 0  |  165.1 req/s  |  ETA: 15.2 min
Scraped 5,000 / 154,714  |  fails: 0  |  166.1 req/s  |  ETA: 15.0 min
  ... flushed partial CSV (5,000 rows)
Scraped 5,500 / 154,714  |  fails: 0  |  167.0 req/s  |  ETA: 14.9 min
Scraped 6,000 / 154,714  |  fails: 0  |  167.8 req/s  |  ETA: 14.8 min
Scraped 6,500 / 154,714  |  fails: 0  |  168.3 req/s  |  ETA: 14.7 min
Scraped 7,000 / 154,714  |  fails: 0  | 

## Step 3 — Save Final CSV

In [10]:
df = pd.DataFrame(results)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

if failed_urls:
    pd.DataFrame(failed_urls, columns=["URL", "Error"]).to_csv(
        FAILED_FILE, index=False, encoding="utf-8-sig"
    )
    print(f"Saved {len(failed_urls):,} failed URLs to '{FAILED_FILE}'")

if os.path.exists(PARTIAL_FILE):
    try:
        os.remove(PARTIAL_FILE)
    except Exception:
        pass

print(f"\nSaved {len(df):,} rows to '{OUTPUT_FILE}'\n")
print("Field completeness:")
print(df.notna().sum())
df.head(5)


Saved 154,714 rows to 'bookxcess_full_inventory.csv'

Field completeness:
Product ID          154714
Title               154714
Product Type        154714
Vendor              154714
SKU                 154712
Current Price       154714
Compare at Price    154714
Currency            154714
Grams               154714
Tags                154714
Body HTML           154707
Image URL           154707
Created At          154714
dtype: int64


,Product ID,Title,Product Type,Vendor,SKU,Current Price,Compare at Price,Currency,Grams,Tags,Body HTML,Image URL,Created At
0,6978035122382,Visions,Kelley Armstrong,Sphere,9780751547238,17.90,62.90,MYR,355,"[2022], BXOS, F CTM, Fiction, Kelley Armstrong...","<p>Olivia Jones is smart, capable, loyal...and...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:08+08:00
1,6978035318990,The Bone Bed,Patricia Cornwell,Sphere,9780751548174,17.90,43.95,MYR,360,"[2022], BXOS, F CTM, Fiction, Paperback, Patri...",<p>A woman has vanished while digging a dinosa...,https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:08+08:00
2,6978035417294,Jasmine Skies,Sita Brahmachari,MacMillan Children's Books,9781447205180,12.90,32.95,MYR,340,"[2022], BXOS, F YA, Fiction, Paperback, RM 10 ...",<p>Mira Levenson is bursting with excitement a...,https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:09+08:00
3,6978034794702,The Old Devils,Kingsley Amis,Vintage Publishing,9780099461050,17.90,59.94,MYR,310,"[2022], BXOS, F LIT, Fiction, Kingsley Amis, P...","<p></p><div style=""text-align: justify;"">Malco...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:07+08:00
4,6978035515598,PHILIP K. DICK'S ELECTRIC DREAMS: VOLUME 1,Philip K. Dick,Gollancz,9781473223288,17.90,53.94,MYR,230,"[2022], BXOS, F SFF, Fiction, Paperback, Phili...",<p>Based on the stories contained in this volu...,https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:09+08:00


## Optional — Re-scrape Failed URLs

Only run this if `failed_urls` has entries you want to recover. Uses half the workers to be gentler on the server.

In [8]:
if failed_urls:
    retry_targets = [u for u, _ in failed_urls]
    recovered     = []
    still_failed  = []

    with ThreadPoolExecutor(max_workers=max(MAX_WORKERS // 2, 5)) as pool:
        futures = {pool.submit(fetch_with_jitter, u): u for u in retry_targets}
        for fut in as_completed(futures):
            url = futures[fut]
            try:
                recovered.append(fut.result())
            except Exception as exc:
                still_failed.append((url, str(exc)))

    print(f"Recovered: {len(recovered):,}   Still failed: {len(still_failed):,}")
    if recovered:
        df_full = pd.concat([df, pd.DataFrame(recovered)], ignore_index=True)
        df_full.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print(f"Updated '{OUTPUT_FILE}' — now {len(df_full):,} rows.")
    if still_failed:
        pd.DataFrame(still_failed, columns=["URL", "Error"]).to_csv(
            FAILED_FILE, index=False, encoding="utf-8-sig"
        )
else:
    print("No failed URLs to retry.")

No failed URLs to retry.
